# Predictive Coding Network — MNIST Demo

Train a small predictive coding network on MNIST using the object API.

**Architecture:**
```
pixels(784) → hidden1(256) → hidden2(64) → class(10)
 Identity      Sigmoid        Sigmoid      Softmax+CE
```

**Expected results:** ~98.14% test accuracy in 20 epochs (~1.37s/epoch on GPU)

For optimizer selection and advanced controls, see `mnist_advanced.py`.

## Imports & JAX Setup

In [1]:
import jax
from fabricpc.nodes import Linear, IdentityNode
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params
from fabricpc.core.activations import SigmoidActivation, SoftmaxActivation
from fabricpc.core.energy import CrossEntropyEnergy
from fabricpc.core.inference import InferenceSGD
from fabricpc.core.initializers import XavierInitializer
import optax
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.utils.data.dataloader import MnistLoader
import time
from fabricpc import setup_jax

setup_jax()  # options: "cpu", "cuda", "tpu"
jax.config.update("jax_default_prng_impl", "threefry2x32")

## Define the Network

Each node declares its shape, activation, and (optionally) energy function.
A single `Edge` between nodes is all that's needed to define a connection —
local derivatives are built in automatically.

In [2]:
pixels = IdentityNode(shape=(784,), name="pixels")
hidden1 = Linear(
    shape=(256,),
    activation=SigmoidActivation(),
    name="hidden1",
    weight_init=XavierInitializer(),
)
hidden2 = Linear(
    shape=(64,),
    activation=SigmoidActivation(),
    name="hidden2",
    weight_init=XavierInitializer(),
)
output = Linear(
    shape=(10,),
    activation=SoftmaxActivation(),
    energy=CrossEntropyEnergy(),
    name="class",
    weight_init=XavierInitializer(),
)

# x= and y= tell the trainer which nodes are inputs and targets
structure = graph(
    nodes=[pixels, hidden1, hidden2, output],
    edges=[
        Edge(source=pixels,  target=hidden1.slot("in")),
        Edge(source=hidden1, target=hidden2.slot("in")),
        Edge(source=hidden2, target=output.slot("in")),
    ],
    task_map=TaskMap(x=pixels, y=output),
    inference=InferenceSGD(eta_infer=0.05, infer_steps=20),
)

## Hyperparameters

In [3]:
train_config = {"num_epochs": 20}
batch_size   = 200
optimizer    = optax.adamw(0.001, weight_decay=0.1)

## Initialize Parameters & Data Loaders

In [4]:
master_rng_key = jax.random.PRNGKey(0)
graph_key, train_key, eval_key = jax.random.split(master_rng_key, 3)

params = initialize_params(structure, graph_key)

train_loader = MnistLoader(
    "train", batch_size=batch_size, tensor_format="flat", shuffle=True, seed=42
)
test_loader = MnistLoader(
    "test", batch_size=batch_size, tensor_format="flat", shuffle=False
)

print(
    f"{len(structure.nodes)} nodes, {len(structure.edges)} edges, "
    f"{sum(p.size for p in jax.tree_util.tree_leaves(params)):,} parameters"
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /home/shamir/tensorflow_datasets/mnist/incomplete.X3J9YJ_3.0.1/mnist-train.tfrecord-[0-9][0-9][0-9][…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /home/shamir/tensorflow_datasets/mnist/incomplete.X3J9YJ_3.0.1/mnist-test.tfrecord-[0-9][0-9][0-9][0…

Dataset mnist downloaded and prepared to /home/shamir/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.
4 nodes, 3 edges, 218,058 parameters


## Train

The first batch triggers JIT compilation — subsequent batches and epochs run at full speed.

In [5]:
print("Training (JIT compilation on first batch)...")
start_time = time.time()
trained_params, energy_history, _ = train_pcn(
    params=params,
    structure=structure,
    train_loader=train_loader,
    optimizer=optimizer,
    config=train_config,
    rng_key=train_key,
    verbose=True,
)
elapsed = time.time() - start_time
print(f"Avg training time: {elapsed / train_config['num_epochs']:.2f}s per epoch")

Training (JIT compilation on first batch)...
Training on 1 device(s): [CudaDevice(id=0)]


  0%|          | 0/6000 [00:00<?, ?it/s]

Avg training time: 3.58s per epoch


## Evaluate

In [6]:
print("Evaluating...")
metrics = evaluate_pcn(
    trained_params, structure, test_loader, train_config, eval_key
)
print(f"Test Accuracy: {metrics['accuracy'] * 100:.2f}%")

Evaluating...
Test Accuracy: 98.13%
